In [1]:
# Imports

# Section geometry
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)

# Materials
from materials.reinforced_concrete.materials import (
    ConcreteMaterial,
    Rebar,
    ShearRebar,
)

# Code checks
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    ShearCheck,
    ShearLoadCase,
)

## Making a Common Concrete Material

In [2]:
# Define concrete material
concrete = ConcreteMaterial(grade="C30/37")

## Common Loadcases

In [3]:
# Define shear load cases
shear_load_cases = [
    {"name": "Support - high shear", "V_Ed": 250, "M_Ed": 50, "N_Ed": 0},
    {"name": "Midspan - low shear", "V_Ed": 80, "M_Ed": 300, "N_Ed": 0},
    {"name": "Column face", "V_Ed": 300, "M_Ed": 100, "N_Ed": 500},
    {"name": "Tension member", "V_Ed": 150, "M_Ed": 100, "N_Ed": -200},
]

## Making a Rectangular Section

In [ ]:
# Section dimensions
width = 530  # mm
height = 530  # mm
cover = 50   # mm (to links, not the longitudinal bars)

# Create section
rect_section = create_rectangular_section(
    width=width,
    height=height,
    section_name=f"{width}x{height} Beam",
)

# Define reinforcement
link_diameter = 16
tension_bar_diameter = 20
compression_bar_diameter = 20

# Calculate bar positions (y-coordinate from bottom)
y_tension = cover + link_diameter + tension_bar_diameter / 2
y_compression = height - cover - link_diameter - compression_bar_diameter / 2

# Side cover for main bars
side_cover = cover + link_diameter

# Tension reinforcement - using create_linear_rebar_layer
tension_rebar = Rebar(diameter=tension_bar_diameter)
num_btm_bars = 5
tension_layer = create_linear_rebar_layer(
    rebar=tension_rebar,
    n_bars=num_btm_bars,
    start_point=(side_cover + tension_bar_diameter / 2, y_tension),
    end_point=(width - side_cover - tension_bar_diameter / 2, y_tension),
)
rect_section.add_rebar_group(tension_layer)

# Compression reinforcement
compression_rebar = Rebar(diameter=compression_bar_diameter)
num_top_bars = 5
compression_layer = create_linear_rebar_layer(
    rebar=compression_rebar,
    n_bars=num_top_bars,
    start_point=(side_cover + compression_bar_diameter / 2, y_compression),
    end_point=(width - side_cover - compression_bar_diameter / 2, y_compression),
)
rect_section.add_rebar_group(compression_layer)

# Shear reinforcement (H10@200, 2-leg links)
shear_rebar = ShearRebar(
    diameter=link_diameter,
    link_spacing=200,
    n_legs=2,
)

rect_section.plot(concrete=concrete)

print(f"Section: {rect_section.section_name}")
print(f"Dimensions: {width}mm x {height}mm")
print(f"Concrete: {concrete.grade} (f_ck = {concrete.f_ck} MPa, f_cd = {concrete.f_cd:.2f} MPa)")
print(f"Tension steel: {num_btm_bars}H{tension_bar_diameter} (A_s = {num_btm_bars * tension_rebar.area:.0f} mm²)")
print(f"Compression steel: {num_btm_bars}H{compression_bar_diameter} (A_s' = {num_top_bars * compression_rebar.area:.0f} mm²)")
print(f"Shear links: H{link_diameter}@{shear_rebar.link_spacing} ({shear_rebar.n_legs}-leg)")
print(f"Reinforcement ratio: {rect_section.reinforcement_ratio:.3%}")

Section: 530x530 Beam
Dimensions: 530mm x 530mm
Concrete: C30/37 (f_ck = 30.0 MPa, f_cd = 20.00 MPa)
Tension steel: 5H20 (A_s = 1571 mm²)
Compression steel: 5H20 (A_s' = 1571 mm²)
Shear links: H16@200.0 (2-leg)
Reinforcement ratio: 1.118%


In [5]:
# Create shear check with rigorous mode (accurate lever arm from force centroids)
shear_check_rect = ShearCheck(
    section=rect_section,
    concrete=concrete,
    shear_reinforcement=shear_rebar,
    use_mechanical_lever_arm=True,  # Use accurate lever arm calculation
)

print(f"Shear Check Configuration")
print(f"=" * 40)
print(f"Web breadth b_w: {shear_check_rect.breadth:.0f} mm")
print(f"Design concrete strength f_cd: {shear_check_rect.f_cd_design:.2f} MPa")
print(f"Design yield strength f_ywd: {shear_check_rect.f_ywd_design:.0f} MPa")
print(f"Mode: {'Rigorous' if shear_check_rect.use_mechanical_lever_arm else 'Approximate'} (z = {'computed' if shear_check_rect.use_mechanical_lever_arm else '0.9d'})")

Shear Check Configuration
Web breadth b_w: 530 mm
Design concrete strength f_cd: 20.00 MPa
Design yield strength f_ywd: 435 MPa
Mode: Rigorous (z = computed)


In [6]:
print("Shear Check Results (EC2 §6.2)")
print("=" * 90)

for lc in shear_load_cases:
    load_case = ShearLoadCase(V_Ed=lc["V_Ed"], M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    result = shear_check_rect.perform_check(
        load_case=load_case,
        #cot_theta_override=1.0,
        )
    
    status = result.status.value.upper()
    d = result.details
    
    print(f"\n{lc['name']}:")
    print(f"  V_Ed = {lc['V_Ed']:5.0f} kN, M_Ed = {lc['M_Ed']:5.0f} kN·m, N_Ed = {lc['N_Ed']:5.0f} kN")
    print(f"  V_Rd,c = {d['V_Rd_c']:5.1f} kN (concrete)")
    print(f"  V_Rd,s = {d['V_Rd_s']:5.1f} kN (reinforcement)")
    print(f"  V_Rd,max = {d['V_Rd_max']:5.1f} kN (strut crushing)")
    print(f"  Governing: {d['governing_mode']}")
    print(f"  θ = {d['theta_deg']:.1f}° (cot θ = {d['cot_theta']:.2f})")
    print(f"  b_w = {d['b_w']:.1f} mm")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Shear Check Results (EC2 §6.2)

Support - high shear:
  V_Ed =   250 kN, M_Ed =    50 kN·m, N_Ed =     0 kN
  V_Rd,c = 129.5 kN (concrete)
  V_Rd,s = 886.1 kN (reinforcement)
  V_Rd,max = 1134.6 kN (strut crushing)
  Governing: shear reinforcement
  θ = 21.8° (cot θ = 2.50)
  b_w = 530.0 mm
  Utilization: 28.2% [PASS]

Midspan - low shear:
  V_Ed =    80 kN, M_Ed =   300 kN·m, N_Ed =     0 kN
  V_Rd,c = 129.5 kN (concrete)
  V_Rd,s = 893.0 kN (reinforcement)
  V_Rd,max = 1143.4 kN (strut crushing)
  Governing: concrete
  θ = 21.8° (cot θ = 2.50)
  b_w = 530.0 mm
  Utilization: 61.8% [PASS]

Column face:
  V_Ed =   300 kN, M_Ed =   100 kN·m, N_Ed =   500 kN
  V_Rd,c = 190.3 kN (concrete)
  V_Rd,s = 774.5 kN (reinforcement)
  V_Rd,max = 1075.2 kN (strut crushing)
  Governing: shear reinforcement
  θ = 21.8° (cot θ = 2.50)
  b_w = 530.0 mm
  Utilization: 38.7% [PASS]

Tension member:
  V_Ed =   150 kN, M_Ed =   100 kN·m, N_Ed =  -200 kN
  V_Rd,c = 105.2 kN (concrete)
  V_Rd,s = 893.0 kN (

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:490: UserWarning:

Lever arm capped: z_mech=419.4mm > 0.9d=408.6mm. Using z=0.9d for EC2 truss model.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:490: UserWarning:

Lever arm capped: z_mech=417.8mm > 0.9d=408.6mm. Using z=0.9d for EC2 truss model.



## Making a Pile Section

In [ ]:
# Section geometry
from materials.reinforced_concrete.geometry import (
    create_circular_section,
    create_circular_perimeter_rebars,
)

# Section parameters
pile_dia = 600       # Diameter (mm)
cover = 50    # Cover to link outer face (mm)
link_dia = 16
bar_dia = 20
n_bars = 10

# Create section (origin at bottom-left, centre at D/2, D/2)
pile_section = create_circular_section(
    diameter=pile_dia,
    section_name=f"{pile_dia}mm Pile",
)

# Add perimeter reinforcement (origin must match section centre)
rebar = Rebar(diameter=bar_dia)
rebar_cover = cover + link_dia
rebar_group = create_circular_perimeter_rebars(
    rebar=rebar,
    diameter=pile_dia,
    cover=rebar_cover,
    n_bars=n_bars,
    origin=(pile_dia / 2, pile_dia / 2),
)
pile_section.add_rebar_group(rebar_group)

# Materials
concrete = ConcreteMaterial(grade="C30/37")
links = ShearRebar(diameter=link_dia, link_spacing=200, n_legs=2, grade="B500B")

# Display
pile_section.plot(concrete=concrete)

print(f"Section: {pile_section.section_name}")
print(f"Diameter: {pile_dia} mm")
print(f"Concrete: {concrete.grade} (f_ck = {concrete.f_ck} MPa, f_cd = {concrete.f_cd:.2f} MPa)")
print(f"Longitudinal steel: {n_bars}H{bar_dia} (A_s = {n_bars * rebar.area:.0f} mm\u00b2)")
print(f"Shear links: H{link_dia}@{links.link_spacing} ({links.n_legs}-leg closed links)")
print(f"Reinforcement ratio: {pile_section.reinforcement_ratio:.3%}")

Section: 600mm Pile
Diameter: 600 mm
Concrete: C30/37 (f_ck = 30.0 MPa, f_cd = 20.00 MPa)
Longitudinal steel: 10H20 (A_s = 3142 mm²)
Shear links: H16@200.0 (2-leg closed links)
Reinforcement ratio: 1.113%


In [8]:
# Create shear check with rigorous mode (accurate lever arm from force centroids)
shear_check_pile = ShearCheck(
    section=pile_section,
    concrete=concrete,
    shear_reinforcement=shear_rebar,
    use_mechanical_lever_arm=True,  # Use accurate lever arm calculation
)

print(f"Shear Check Configuration")
print(f"=" * 40)
print(f"Web breadth b_w: {shear_check_pile.breadth:.0f} mm")
print(f"Design concrete strength f_cd: {shear_check_pile.f_cd_design:.2f} MPa")
print(f"Design yield strength f_ywd: {shear_check_pile.f_ywd_design:.0f} MPa")
print(f"Mode: {'Rigorous' if shear_check_pile.use_mechanical_lever_arm else 'Approximate'} (z = {'computed' if shear_check_pile.use_mechanical_lever_arm else '0.9d'})")

Shear Check Configuration
Web breadth b_w: 165 mm
Design concrete strength f_cd: 20.00 MPa
Design yield strength f_ywd: 435 MPa
Mode: Rigorous (z = computed)


In [9]:
print("Shear Check Results (EC2 §6.2)")
print("=" * 90)

for lc in shear_load_cases:
    load_case = ShearLoadCase(V_Ed=lc["V_Ed"], M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    result = shear_check_pile.perform_check(
        load_case=load_case,
        #cot_theta_override=1.0,        
        )
    
    status = result.status.value.upper()
    d = result.details
    
    print(f"\n{lc['name']}:")
    print(f"  V_Ed = {lc['V_Ed']:5.0f} kN, M_Ed = {lc['M_Ed']:5.0f} kN·m, N_Ed = {lc['N_Ed']:5.0f} kN")
    print(f"  V_Rd,c = {d['V_Rd_c']:5.1f} kN (concrete)")
    print(f"  V_Rd,s = {d['V_Rd_s']:5.1f} kN (reinforcement)")
    print(f"  V_Rd,max = {d['V_Rd_max']:5.1f} kN (strut crushing)")
    print(f"  Governing: {d['governing_mode']}")
    print(f"  θ = {d['theta_deg']:.1f}° (cot θ = {d['cot_theta']:.2f})")
    print(f"  b_w = {d['b_w']:.1f} mm")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Shear Check Results (EC2 §6.2)

Support - high shear:
  V_Ed =   250 kN, M_Ed =    50 kN·m, N_Ed =     0 kN
  V_Rd,c =  54.6 kN (concrete)
  V_Rd,s = 696.8 kN (reinforcement)
  V_Rd,max = 325.7 kN (strut crushing)
  Governing: compression strut
  θ = 25.1° (cot θ = 2.14)
  b_w = 165.4 mm
  Utilization: 76.8% [PASS]

Midspan - low shear:
  V_Ed =    80 kN, M_Ed =   300 kN·m, N_Ed =     0 kN
  V_Rd,c =  54.6 kN (concrete)
  V_Rd,s = 696.2 kN (reinforcement)
  V_Rd,max = 278.2 kN (strut crushing)
  Governing: compression strut
  θ = 21.8° (cot θ = 2.50)
  b_w = 165.4 mm
  Utilization: 28.8% [PASS]

Column face:
  V_Ed =   300 kN, M_Ed =   100 kN·m, N_Ed =   500 kN
  V_Rd,c =  70.3 kN (concrete)
  V_Rd,s = 482.5 kN (reinforcement)
  V_Rd,max = 331.5 kN (strut crushing)
  Governing: compression strut
  θ = 32.4° (cot θ = 1.58)
  b_w = 165.4 mm
  Utilization: 90.5% [PASS]

Tension member:
  V_Ed =   150 kN, M_Ed =   100 kN·m, N_Ed =  -200 kN
  V_Rd,c =  47.7 kN (concrete)
  V_Rd,s = 809.8 kN